In [ ]:
'''%pip install --upgrade pip
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
%pip install transformers accelerate datasets peft bitsandbytes wandb
%pip install langchain
%pip install langchain_huggingface'''

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import shutil
import json
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    default_data_collator
)

In [ ]:
dataset_path = "../evaluating_text2cypher/FTdataset_local.json"
assert os.path.exists(dataset_path), f"{dataset_path} not found"
with open(dataset_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)
print(f"Finetuning dataset size: {len(train_data)}")

In [ ]:
MODEL_NAME = "meta-llama/Llama-3.1-8B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    load_in_4bit=True,
    torch_dtype=torch.bfloat16,
    token=True
)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

pairs = [
    {"prompt": f"{row['question']}\nCypher query:",
     "completion": " " + row["cypher"]}
    for row in train_data
]

dataset = Dataset.from_list(pairs)

def preprocess_function(example):
    text = example["prompt"] + example["completion"]
    encoding = tokenizer(
        text,
        max_length=512,
        truncation=True,
        padding="max_length"
    )
    labels = encoding["input_ids"].copy()

    # Mask padding
    labels = [tok if tok != tokenizer.pad_token_id else -100 for tok in labels]

    prompt_len = len(tokenizer(example["prompt"])["input_ids"])
    labels[:prompt_len] = [-100] * prompt_len

    encoding["labels"] = labels
    return encoding

tokenized_dataset = dataset.map(preprocess_function, remove_columns=["prompt", "completion"])
print("Tokenized dataset sample:", tokenized_dataset[0])

In [ ]:
data_collator = default_data_collator

training_args = TrainingArguments(
    output_dir="./results-llama3-lora",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    eval_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()

In [ ]:
lora_save_path = "./llama3_lora_mirnakgt2c"
model.save_pretrained(lora_save_path)
tokenizer.save_pretrained(lora_save_path)

****

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftModel
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.1-8B",
    device_map="auto",
    load_in_4bit=True,
    torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained("./llama3_lora_mirnakgt2c")
model = PeftModel.from_pretrained(base_model, "./llama3_lora_mirnakgt2c")

hf_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    temperature=0.0,
    top_p=0.95,
    do_sample=False
)
llm = HuggingFacePipeline(pipeline=hf_pipe)

cypher_template = """Write a Cypher query that would answer the user's question.
Return only Cypher statement, no backticks, nothing else.

Question: {question}
Cypher query:"""
cypher_prompt = ChatPromptTemplate.from_template(cypher_template)

cypher_chain = cypher_prompt | llm | StrOutputParser()

In [ ]:
response = cypher_chain.invoke(
    {
        "question": "How many miRNAs have the keyword 'precursor' in the label and a sequence size under 100 nucleotides?"
    }
).rsplit("\nCypher query: ", 1)[-1]

print(response)